# Tail Risk Hedging & VIX Strategies

**CQF-Level Example**: Professional tail risk management:
- VIX term structure and contango/backwardation
- Put protection overlay strategies
- Tail risk parity allocation
- Dynamic hedging based on regime
- Cost-efficient protection structures

**Connectors Used:**
- `qj.cboe` - VIX indices and term structure
- `qj.eod` - ETF and equity prices
- `qj.fred` - Economic indicators

**API:** https://api.quantjourney.cloud

## Run Output

![43_tail_risk_hedging](../plots/43_tail_risk_hedging_output_01.png)

**Prepared by QuantJourney.** Candidate notebook source is kept clean and unexecuted. Generated run artifacts are committed under `plots/` and indexed in `plots/manifest.json`.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import norm
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "png"

from quantjourney.sdk import QuantJourneyAPI

import os
API_KEY = os.environ.get("QJ_API_KEY", "qj_...")

qj = QuantJourneyAPI(api_key=API_KEY)
print("✓ Connected to QuantJourney API")


## 1. Fetch VIX Data

In [ ]:
# VIX indices from CBOE
vix_symbols = {
    'VIX': 'VIX Index (30-day)',
    'VIX9D': 'VIX 9-Day',
    'VIX3M': 'VIX 3-Month',
    'VIX6M': 'VIX 6-Month',
    'VVIX': 'VIX of VIX'
}

start_date = '2015-01-01'
end_date = '2024-12-31'

vix_data = {}

# Fetch VIX data using dedicated method
try:
    response = qj.cboe.get_vix_data(
        start_date=start_date,
        end_date=end_date,
        interval='1d'
    )
    data = response.get('value', response) if isinstance(response, dict) else response
    if isinstance(data, list) and len(data) > 0:
        df = pd.DataFrame(data)
        df['date'] = pd.to_datetime(df['date'])
        df = df.set_index('date')
        vix_data['VIX'] = df['close']
        print(f"✓ VIX: VIX Index (30-day)")
    elif data is not None:
        print(f"⚠ VIX: No data returned")
except Exception as e:
    print(f"✗ VIX: {e}")

# Fetch VVIX data using dedicated method
try:
    response = qj.cboe.get_vvix_data(
        start_date=start_date,
        end_date=end_date,
        interval='1d'
    )
    data = response.get('value', response) if isinstance(response, dict) else response
    if isinstance(data, list) and len(data) > 0:
        df = pd.DataFrame(data)
        df['date'] = pd.to_datetime(df['date'])
        df = df.set_index('date')
        vix_data['VVIX'] = df['close']
        print(f"✓ VVIX: VIX of VIX")
    elif data is not None:
        print(f"⚠ VVIX: No data returned")
except Exception as e:
    print(f"✗ VVIX: {e}")

# Fetch VIX term structure (VIX9D, VIX3M, VIX6M)
try:
    response = qj.cboe.get_vix_term_structure(
        start_date=start_date,
        end_date=end_date
    )
    data = response.get('value', response) if isinstance(response, dict) else response
    if isinstance(data, list) and len(data) > 0:
        df = pd.DataFrame(data)
        df['date'] = pd.to_datetime(df['date'])
        df = df.set_index('date')
        # Extract term structure components if available
        for col in ['vix9d', 'vix3m', 'vix6m', 'VIX9D', 'VIX3M', 'VIX6M']:
            if col in df.columns:
                vix_data[col.upper()] = df[col]
                print(f"✓ {col.upper()}: VIX Term Structure")
    elif data is not None:
        print(f"⚠ VIX Term Structure: No data returned")
except Exception as e:
    print(f"✗ VIX Term Structure: {e}")

# Fallback synthetic VIX
if len(vix_data) < 2:
    print("\nGenerating synthetic VIX data...")
    dates = pd.date_range(start=start_date, end=end_date, freq='B')
    np.random.seed(42)
    
    # VIX mean-reverting with jumps
    vix = [15.0]
    for i in range(1, len(dates)):
        # Mean reversion to 18
        drift = 0.02 * (18 - vix[-1])
        # Volatility clustering
        vol = 0.8 + 0.4 * np.abs(vix[-1] - 18) / 10
        # Random jump (spikes)
        jump = 0
        if np.random.rand() < 0.01:  # 1% chance of spike
            jump = np.random.exponential(10)
        vix.append(max(9, vix[-1] + drift + vol * np.random.randn() + jump))
    
    vix_data['VIX'] = pd.Series(vix, index=dates)
    vix_data['VIX9D'] = pd.Series([v * (0.95 + 0.1*np.random.rand()) for v in vix], index=dates)
    vix_data['VIX3M'] = pd.Series([v * (1.05 + 0.05*np.random.rand()) for v in vix], index=dates)  # Usually in contango
    vix_data['VIX6M'] = pd.Series([v * (1.08 + 0.05*np.random.rand()) for v in vix], index=dates)
    vix_data['VVIX'] = pd.Series([80 + 30*v/20 + np.random.randn()*5 for v in vix], index=dates)
    print("✓ Synthetic VIX data generated")


In [ ]:
# Fetch S&P 500 for correlation analysis
try:
    response = qj.eod.get_historical_prices(
        symbol='SPY',
        start_date=start_date,
        end_date=end_date,
        frequency='1d'
    )
    data = response.get('value', response) if isinstance(response, dict) else response
    if isinstance(data, list) and len(data) > 0:
        df = pd.DataFrame(data)
        df['date'] = pd.to_datetime(df['date'])
        df = df.set_index('date')
        spy_prices = df['adjusted_close' if 'adjusted_close' in df.columns else 'close']
        print(f"✓ SPY data loaded")
except Exception as e:
    print(f"✗ SPY: {e}")
    dates = pd.date_range(start=start_date, end=end_date, freq='B')
    np.random.seed(123)
    spy_ret = 0.10/252 + 0.16/np.sqrt(252) * np.random.randn(len(dates))
    spy_prices = pd.Series(100 * np.cumprod(1 + spy_ret), index=dates)
    print("✓ Synthetic SPY generated")


In [ ]:
# Build aligned dataframes
vix_df = pd.DataFrame(vix_data)

# Ensure all columns are numeric
for col in vix_df.columns:
    vix_df[col] = pd.to_numeric(vix_df[col], errors='coerce')

vix_df = vix_df.ffill().dropna()

# Align with SPY
common_idx = vix_df.index.intersection(spy_prices.index)
vix_df = vix_df.loc[common_idx]
spy_prices = spy_prices.loc[common_idx]
spy_returns = spy_prices.pct_change().dropna()

print(f"\nData period: {vix_df.index[0].date()} to {vix_df.index[-1].date()}")
print(f"\nLatest VIX readings:")
print(vix_df.iloc[-1].round(2))


## 2. VIX Term Structure Analysis

In [ ]:
# Calculate term structure metrics
def analyze_vix_term_structure(vix_df):
    """
    Analyze VIX term structure:
    - Contango: VIX3M > VIX (normal, roll cost)
    - Backwardation: VIX > VIX3M (fear, hedging demand)
    """
    ts = pd.DataFrame(index=vix_df.index)
    
    if 'VIX' in vix_df.columns and 'VIX3M' in vix_df.columns:
        # Term spread (3M - spot)
        ts['spread_3m'] = vix_df['VIX3M'] - vix_df['VIX']
        ts['spread_pct'] = ts['spread_3m'] / vix_df['VIX'] * 100
        
        # Contango/Backwardation flag
        ts['contango'] = ts['spread_3m'] > 0
        
        # Roll yield (annualized, negative in contango)
        ts['roll_yield'] = -ts['spread_pct'] * 4  # Approximate annual
    
    if 'VIX9D' in vix_df.columns and 'VIX' in vix_df.columns:
        # Short-term vs spot (fear gauge)
        ts['short_spread'] = vix_df['VIX9D'] - vix_df['VIX']
    
    return ts

term_structure = analyze_vix_term_structure(vix_df)

# Stats
if 'contango' in term_structure.columns:
    contango_pct = term_structure['contango'].mean() * 100
    print(f"\nTerm Structure Analysis:")
    print(f"  Contango frequency: {contango_pct:.1f}%")
    print(f"  Current spread (3M-spot): {term_structure['spread_3m'].iloc[-1]:.2f}")
    print(f"  Current status: {'Contango' if term_structure['contango'].iloc[-1] else 'Backwardation'}")


In [ ]:
# Term structure visualization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['VIX Spot vs 3-Month', 'Term Structure Spread',
                    'VIX Level Distribution', 'VIX-SPY Correlation'],
    vertical_spacing=0.12
)

# 1. VIX levels
fig.add_trace(
    go.Scatter(x=vix_df.index, y=vix_df['VIX'], name='VIX',
               line=dict(color='orange')),
    row=1, col=1
)
if 'VIX3M' in vix_df.columns:
    fig.add_trace(
        go.Scatter(x=vix_df.index, y=vix_df['VIX3M'], name='VIX3M',
                   line=dict(color='cyan')),
        row=1, col=1
    )

# 2. Spread with contango/backwardation coloring
if 'spread_3m' in term_structure.columns:
    colors = ['green' if x > 0 else 'red' for x in term_structure['spread_3m']]
    fig.add_trace(
        go.Scatter(x=term_structure.index, y=term_structure['spread_3m'],
                   mode='lines', name='Spread',
                   line=dict(color='white', width=1)),
        row=1, col=2
    )
    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=2)

# 3. VIX distribution
fig.add_trace(
    go.Histogram(x=vix_df['VIX'], name='VIX Dist',
                 marker_color='orange', opacity=0.7),
    row=2, col=1
)
fig.add_vline(x=vix_df['VIX'].median(), line_dash='dash', line_color='white', row=2, col=1)

# 4. Rolling correlation VIX-SPY
vix_ret = vix_df['VIX'].pct_change()
rolling_corr = vix_ret.rolling(63).corr(spy_returns)
fig.add_trace(
    go.Scatter(x=rolling_corr.index, y=rolling_corr, name='Correlation',
               line=dict(color='magenta')),
    row=2, col=2
)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=2)

fig.update_layout(
    title='VIX Term Structure Analysis',
    template='plotly_dark',
    height=600
)
fig.show()


## 3. Tail Risk Metrics

In [ ]:
def calculate_tail_risk_metrics(returns, vix_series, window=252):
    """
    Calculate tail risk metrics:
    - Historical VaR and CVaR
    - Tail risk ratio (implied vs realized)
    - Skewness and kurtosis
    - Max drawdown
    """
    metrics = pd.DataFrame(index=returns.index[window:])
    
    for i in range(window, len(returns)):
        ret_window = returns.iloc[i-window:i]
        date = returns.index[i]
        
        # VaR (95%)
        var_95 = np.percentile(ret_window, 5)
        
        # CVaR (Expected Shortfall)
        cvar_95 = ret_window[ret_window <= var_95].mean()
        
        # Realized volatility (annualized)
        realized_vol = ret_window.std() * np.sqrt(252) * 100
        
        # Skewness and kurtosis
        skew = stats.skew(ret_window)
        kurt = stats.kurtosis(ret_window)
        
        # Drawdown
        cum_ret = (1 + ret_window).cumprod()
        max_dd = (cum_ret / cum_ret.cummax() - 1).min()
        
        metrics.loc[date, 'VaR_95'] = var_95 * 100
        metrics.loc[date, 'CVaR_95'] = cvar_95 * 100 if not np.isnan(cvar_95) else var_95 * 100
        metrics.loc[date, 'Realized_Vol'] = realized_vol
        metrics.loc[date, 'Skewness'] = skew
        metrics.loc[date, 'Kurtosis'] = kurt
        metrics.loc[date, 'Max_DD'] = max_dd * 100
    
    # Implied vs Realized ratio
    vix_aligned = vix_series.reindex(metrics.index, method='ffill')
    metrics['VRP'] = vix_aligned - metrics['Realized_Vol']  # Volatility Risk Premium
    metrics['Tail_Ratio'] = metrics['CVaR_95'] / metrics['VaR_95']
    
    return metrics

# Calculate tail metrics
tail_metrics = calculate_tail_risk_metrics(spy_returns, vix_df['VIX'])

print("\nCurrent Tail Risk Metrics:")
print(f"  VaR (95%, 1-day):    {tail_metrics['VaR_95'].iloc[-1]:.2f}%")
print(f"  CVaR (95%, 1-day):   {tail_metrics['CVaR_95'].iloc[-1]:.2f}%")
print(f"  Tail Ratio:          {tail_metrics['Tail_Ratio'].iloc[-1]:.2f}")
print(f"  Vol Risk Premium:    {tail_metrics['VRP'].iloc[-1]:.2f}")
print(f"  Skewness:            {tail_metrics['Skewness'].iloc[-1]:.2f}")
print(f"  Excess Kurtosis:     {tail_metrics['Kurtosis'].iloc[-1]:.2f}")


In [ ]:
# Tail risk dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['VaR and CVaR (95%)', 'Volatility Risk Premium',
                    'Return Distribution vs Normal', 'Rolling Skewness'],
    vertical_spacing=0.12
)

# 1. VaR and CVaR
fig.add_trace(
    go.Scatter(x=tail_metrics.index, y=tail_metrics['VaR_95'],
               name='VaR', line=dict(color='orange')),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=tail_metrics.index, y=tail_metrics['CVaR_95'],
               name='CVaR', line=dict(color='red')),
    row=1, col=1
)

# 2. VRP
colors_vrp = ['green' if x > 0 else 'red' for x in tail_metrics['VRP']]
fig.add_trace(
    go.Scatter(x=tail_metrics.index, y=tail_metrics['VRP'],
               name='VRP', line=dict(color='cyan'), fill='tozeroy'),
    row=1, col=2
)
fig.add_hline(y=0, line_dash='dash', line_color='white', row=1, col=2)

# 3. Return distribution vs normal
ret_std = spy_returns.std()
ret_mean = spy_returns.mean()
x_range = np.linspace(-5*ret_std, 5*ret_std, 100)
normal_pdf = norm.pdf(x_range, ret_mean, ret_std)

fig.add_trace(
    go.Histogram(x=spy_returns, name='Actual', histnorm='probability density',
                 marker_color='cyan', opacity=0.7),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=x_range, y=normal_pdf, name='Normal',
               line=dict(color='white', dash='dash')),
    row=2, col=1
)

# 4. Rolling skewness
fig.add_trace(
    go.Scatter(x=tail_metrics.index, y=tail_metrics['Skewness'],
               name='Skewness', line=dict(color='magenta')),
    row=2, col=2
)
fig.add_hline(y=0, line_dash='dash', line_color='white', row=2, col=2)

fig.update_layout(
    title='Tail Risk Dashboard',
    template='plotly_dark',
    height=600
)
fig.show()


## 4. Put Protection Strategies

In [ ]:
def black_scholes_put(S, K, T, r, sigma):
    """
    Black-Scholes put option price
    """
    if T <= 0 or sigma <= 0:
        return max(K - S, 0)
    
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    
    put_price = K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)
    return put_price


def simulate_put_protection(prices, vix_series, 
                           strike_pct=0.95,  # 5% OTM put
                           roll_freq=21,     # Monthly roll
                           hedge_ratio=1.0):  # Full hedge
    """
    Simulate rolling put protection strategy:
    - Buy puts at specified strike
    - Roll monthly
    - Track cost and payoff
    """
    results = []
    
    r = 0.02  # Risk-free rate
    T = roll_freq / 252  # Time to expiry
    
    i = 0
    while i < len(prices) - roll_freq:
        date = prices.index[i]
        S = prices.iloc[i]
        K = S * strike_pct  # Strike price
        
        # Use VIX as implied vol (convert from % to decimal)
        sigma = vix_series.loc[date] / 100 if date in vix_series.index else 0.20
        
        # Put cost
        put_cost = black_scholes_put(S, K, T, r, sigma)
        put_cost_pct = put_cost / S * 100  # As % of portfolio
        
        # Price at expiry
        S_expiry = prices.iloc[i + roll_freq]
        
        # Put payoff
        put_payoff = max(K - S_expiry, 0)
        put_payoff_pct = put_payoff / S * 100
        
        # Portfolio return (unhedged)
        portfolio_ret = (S_expiry / S - 1) * 100
        
        # Hedged return
        hedged_ret = portfolio_ret + hedge_ratio * (put_payoff_pct - put_cost_pct)
        
        results.append({
            'date': date,
            'spot': S,
            'strike': K,
            'vix': sigma * 100,
            'put_cost_pct': put_cost_pct,
            'put_payoff_pct': put_payoff_pct,
            'unhedged_ret': portfolio_ret,
            'hedged_ret': hedged_ret,
            'protection_value': put_payoff_pct - put_cost_pct
        })
        
        i += roll_freq
    
    return pd.DataFrame(results).set_index('date')

# Simulate different strike levels
protection_strategies = {}
for strike in [0.90, 0.95, 1.00]:
    protection_strategies[f'{int(strike*100)}% Strike'] = simulate_put_protection(
        spy_prices, vix_df['VIX'], strike_pct=strike
    )

print("\nPut Protection Strategy Summary:")
for name, df in protection_strategies.items():
    avg_cost = df['put_cost_pct'].mean()
    avg_payoff = df['put_payoff_pct'].mean()
    net_cost = avg_cost - avg_payoff
    print(f"\n  {name}:")
    print(f"    Avg Monthly Cost:   {avg_cost:.2f}%")
    print(f"    Avg Monthly Payoff: {avg_payoff:.2f}%")
    print(f"    Net Cost (annual):  {net_cost*12:.2f}%")


In [ ]:
# Compare hedged vs unhedged performance
strategy_95 = protection_strategies['95% Strike']

# Cumulative returns
cum_unhedged = (1 + strategy_95['unhedged_ret']/100).cumprod()
cum_hedged = (1 + strategy_95['hedged_ret']/100).cumprod()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Cumulative Performance', 'Protection Cost vs VIX',
                    'Monthly Returns Comparison', 'Drawdown Comparison'],
    vertical_spacing=0.12
)

# 1. Cumulative performance
fig.add_trace(
    go.Scatter(x=cum_unhedged.index, y=cum_unhedged, name='Unhedged',
               line=dict(color='orange')),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=cum_hedged.index, y=cum_hedged, name='95% Put Hedge',
               line=dict(color='cyan')),
    row=1, col=1
)

# 2. Cost vs VIX
fig.add_trace(
    go.Scatter(x=strategy_95['vix'], y=strategy_95['put_cost_pct'],
               mode='markers', name='Cost vs VIX',
               marker=dict(color='red', size=5)),
    row=1, col=2
)

# 3. Monthly returns
fig.add_trace(
    go.Scatter(x=strategy_95['unhedged_ret'], y=strategy_95['hedged_ret'],
               mode='markers', name='Returns',
               marker=dict(color='magenta', size=5)),
    row=2, col=1
)
# Add 45-degree line
ret_range = [-15, 15]
fig.add_trace(
    go.Scatter(x=ret_range, y=ret_range, mode='lines',
               line=dict(color='white', dash='dash'), showlegend=False),
    row=2, col=1
)

# 4. Drawdowns
dd_unhedged = cum_unhedged / cum_unhedged.cummax() - 1
dd_hedged = cum_hedged / cum_hedged.cummax() - 1

fig.add_trace(
    go.Scatter(x=dd_unhedged.index, y=dd_unhedged*100, name='Unhedged DD',
               line=dict(color='orange')),
    row=2, col=2
)
fig.add_trace(
    go.Scatter(x=dd_hedged.index, y=dd_hedged*100, name='Hedged DD',
               line=dict(color='cyan')),
    row=2, col=2
)

fig.update_xaxes(title_text='Unhedged Return (%)', row=2, col=1)
fig.update_yaxes(title_text='Hedged Return (%)', row=2, col=1)

fig.update_layout(
    title='Put Protection Analysis (95% Strike, Monthly Roll)',
    template='plotly_dark',
    height=600
)
fig.show()


## 5. Dynamic Hedging Strategy

In [ ]:
def dynamic_hedge_strategy(prices, vix_series, tail_metrics,
                          vix_threshold_low=15,
                          vix_threshold_high=25,
                          vrp_threshold=3):
    """
    Dynamic hedging strategy that adjusts protection based on:
    - VIX level (cheap/expensive protection)
    - VRP (volatility risk premium)
    - Term structure (contango/backwardation)
    
    Hedge Ratio:
    - Low VIX (<15): Full hedge (100%) - protection is cheap
    - Medium VIX (15-25): Partial hedge (50%)
    - High VIX (>25): Minimal hedge (25%) - protection expensive
    - Positive VRP adjustment: Reduce hedge when VRP high
    """
    results = []
    r = 0.02
    T = 21/252  # Monthly
    
    i = 0
    while i < len(prices) - 21:
        date = prices.index[i]
        S = prices.iloc[i]
        
        # Current VIX
        vix = vix_series.loc[date] if date in vix_series.index else 20
        
        # Current VRP
        vrp = tail_metrics.loc[date, 'VRP'] if date in tail_metrics.index else 0
        
        # Determine hedge ratio
        if vix < vix_threshold_low:
            base_hedge = 1.0  # Full hedge
            regime = 'Low Vol'
        elif vix < vix_threshold_high:
            base_hedge = 0.5  # Partial
            regime = 'Medium Vol'
        else:
            base_hedge = 0.25  # Minimal
            regime = 'High Vol'
        
        # VRP adjustment
        if vrp > vrp_threshold:
            hedge_ratio = base_hedge * 0.8  # Reduce when VRP high (expensive)
        else:
            hedge_ratio = base_hedge
        
        # Calculate protection
        K = S * 0.95
        sigma = vix / 100
        put_cost = black_scholes_put(S, K, T, r, sigma)
        put_cost_pct = put_cost / S * 100 * hedge_ratio
        
        # Expiry
        S_expiry = prices.iloc[i + 21]
        put_payoff = max(K - S_expiry, 0)
        put_payoff_pct = put_payoff / S * 100 * hedge_ratio
        
        # Returns
        unhedged_ret = (S_expiry / S - 1) * 100
        hedged_ret = unhedged_ret + put_payoff_pct - put_cost_pct
        
        results.append({
            'date': date,
            'vix': vix,
            'vrp': vrp,
            'regime': regime,
            'hedge_ratio': hedge_ratio,
            'cost_pct': put_cost_pct,
            'unhedged_ret': unhedged_ret,
            'hedged_ret': hedged_ret
        })
        
        i += 21
    
    return pd.DataFrame(results).set_index('date')

# Run dynamic strategy
dynamic_results = dynamic_hedge_strategy(spy_prices, vix_df['VIX'], tail_metrics)

print("\nDynamic Hedging Strategy Summary:")
print(f"\n  Regime Distribution:")
print(dynamic_results['regime'].value_counts())
print(f"\n  Average Hedge Ratio: {dynamic_results['hedge_ratio'].mean():.1%}")
print(f"  Average Monthly Cost: {dynamic_results['cost_pct'].mean():.2f}%")


In [ ]:
# Compare strategies
static_results = protection_strategies['95% Strike']

# Align dates
common_dates = dynamic_results.index.intersection(static_results.index)

comparison = pd.DataFrame({
    'Unhedged': static_results.loc[common_dates, 'unhedged_ret'],
    'Static 95% Put': static_results.loc[common_dates, 'hedged_ret'],
    'Dynamic Hedge': dynamic_results.loc[common_dates, 'hedged_ret']
})

# Cumulative
cum_comparison = (1 + comparison/100).cumprod()

fig = go.Figure()

for col in cum_comparison.columns:
    fig.add_trace(go.Scatter(
        x=cum_comparison.index, y=cum_comparison[col],
        mode='lines', name=col
    ))

fig.update_layout(
    title='Strategy Comparison: Unhedged vs Static vs Dynamic Hedge',
    xaxis_title='Date',
    yaxis_title='Cumulative Return',
    template='plotly_dark',
    height=450
)
fig.show()


In [ ]:
# Performance metrics comparison
def calc_strategy_metrics(returns):
    cum = (1 + returns/100).cumprod()
    total_ret = cum.iloc[-1] - 1
    ann_ret = (1 + total_ret) ** (12/len(returns)) - 1
    ann_vol = returns.std() * np.sqrt(12) / 100
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    max_dd = (cum / cum.cummax() - 1).min()
    
    return {
        'Total Return': total_ret * 100,
        'Annual Return': ann_ret * 100,
        'Annual Vol': ann_vol * 100,
        'Sharpe': sharpe,
        'Max DD': max_dd * 100
    }

print("\n" + "="*70)
print("STRATEGY PERFORMANCE COMPARISON")
print("="*70)

metrics_comparison = pd.DataFrame({
    col: calc_strategy_metrics(comparison[col]) for col in comparison.columns
}).T

print(f"\n{'Metric':<20} {'Unhedged':>12} {'Static':>12} {'Dynamic':>12}")
print("-"*60)
for metric in metrics_comparison.columns:
    row = metrics_comparison[metric]
    if 'Return' in metric or 'Vol' in metric or 'DD' in metric:
        print(f"{metric:<20} {row['Unhedged']:>11.1f}% {row['Static 95% Put']:>11.1f}% {row['Dynamic Hedge']:>11.1f}%")
    else:
        print(f"{metric:<20} {row['Unhedged']:>12.2f} {row['Static 95% Put']:>12.2f} {row['Dynamic Hedge']:>12.2f}")


## 6. Tail Risk Parity

In [ ]:
def tail_risk_parity_weights(returns_df, target_cvar=0.02):
    """
    Calculate tail risk parity weights:
    - Equal contribution to portfolio CVaR
    - Assets with higher tail risk get lower weights
    
    Simplified approach: inverse CVaR weighting
    """
    cvars = {}
    
    for col in returns_df.columns:
        ret = returns_df[col].dropna()
        var_95 = np.percentile(ret, 5)
        cvar_95 = ret[ret <= var_95].mean()
        cvars[col] = abs(cvar_95)  # Use absolute value
    
    # Inverse CVaR weights
    total_inv_cvar = sum(1/v for v in cvars.values())
    weights = {k: (1/v) / total_inv_cvar for k, v in cvars.items()}
    
    return weights, cvars

# Create multi-asset portfolio for demonstration
# Fetch additional ETFs
assets = ['SPY', 'TLT', 'GLD', 'EFA']
asset_prices = {}

for symbol in assets:
    try:
        response = qj.eod.get_historical_prices(
            symbol=symbol,
            start_date=start_date,
            end_date=end_date,
            frequency='1d'
        )
        data = response.get('value', response) if isinstance(response, dict) else response
        if isinstance(data, list) and len(data) > 0:
            df = pd.DataFrame(data)
            df['date'] = pd.to_datetime(df['date'])
            df = df.set_index('date')
            asset_prices[symbol] = df['adjusted_close' if 'adjusted_close' in df.columns else 'close']
            print(f"✓ {symbol}")
    except Exception as e:
        print(f"✗ {symbol}: {e}")

# Fallback
if len(asset_prices) < 4:
    print("\nUsing synthetic multi-asset data...")
    dates = spy_prices.index
    np.random.seed(555)
    
    asset_prices['SPY'] = spy_prices
    # Bonds (negative correlation with stocks)
    bond_ret = 0.04/252 + 0.08/np.sqrt(252) * (-0.3 * spy_returns.values + 0.95 * np.random.randn(len(spy_returns)))
    asset_prices['TLT'] = pd.Series(100 * np.cumprod(np.concatenate([[1], 1 + bond_ret])), index=dates)
    # Gold (low correlation)
    gold_ret = 0.05/252 + 0.15/np.sqrt(252) * np.random.randn(len(spy_returns))
    asset_prices['GLD'] = pd.Series(100 * np.cumprod(np.concatenate([[1], 1 + gold_ret])), index=dates)
    # International (high correlation)
    intl_ret = 0.07/252 + 0.18/np.sqrt(252) * (0.7 * spy_returns.values + 0.7 * np.random.randn(len(spy_returns)))
    asset_prices['EFA'] = pd.Series(100 * np.cumprod(np.concatenate([[1], 1 + intl_ret])), index=dates)


In [ ]:
# Build asset returns
asset_prices_df = pd.DataFrame(asset_prices).dropna()
asset_returns_df = asset_prices_df.pct_change().dropna()

# Calculate tail risk parity weights
trp_weights, asset_cvars = tail_risk_parity_weights(asset_returns_df)

# Equal weight for comparison
equal_weights = {k: 1/len(assets) for k in assets if k in asset_returns_df.columns}

print("\nTail Risk Parity Allocation:")
print(f"\n{'Asset':<10} {'CVaR (95%)':<15} {'TRP Weight':<15} {'Equal Weight':<15}")
print("-"*55)
for asset in trp_weights:
    print(f"{asset:<10} {asset_cvars[asset]*100:>12.2f}% {trp_weights[asset]*100:>12.1f}% {equal_weights.get(asset,0)*100:>12.1f}%")


In [ ]:
# Compare TRP vs Equal weight portfolios
trp_returns = (asset_returns_df * pd.Series(trp_weights)).sum(axis=1)
equal_returns = (asset_returns_df * pd.Series(equal_weights)).sum(axis=1)

cum_trp = (1 + trp_returns).cumprod()
cum_equal = (1 + equal_returns).cumprod()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Cumulative Performance', 'Allocation Comparison']
)

# Performance
fig.add_trace(
    go.Scatter(x=cum_trp.index, y=cum_trp, name='Tail Risk Parity',
               line=dict(color='cyan')),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=cum_equal.index, y=cum_equal, name='Equal Weight',
               line=dict(color='orange')),
    row=1, col=1
)

# Weights comparison
fig.add_trace(
    go.Bar(x=list(trp_weights.keys()), y=[v*100 for v in trp_weights.values()],
           name='TRP', marker_color='cyan'),
    row=1, col=2
)
fig.add_trace(
    go.Bar(x=list(equal_weights.keys()), y=[v*100 for v in equal_weights.values()],
           name='Equal', marker_color='orange'),
    row=1, col=2
)

fig.update_layout(
    title='Tail Risk Parity vs Equal Weight',
    template='plotly_dark',
    height=400,
    barmode='group'
)
fig.update_yaxes(title_text='Weight (%)', row=1, col=2)
fig.show()


## Summary

This CQF-level tail risk management example covered:

1. **VIX Analysis**: Term structure, contango/backwardation, roll dynamics
2. **Tail Risk Metrics**: VaR, CVaR, skewness, kurtosis, tail ratios
3. **Put Protection**: Static rolling put strategies at different strikes
4. **Dynamic Hedging**: Regime-based hedge ratio adjustment
5. **Tail Risk Parity**: Equal tail risk contribution allocation
6. **Cost Analysis**: Protection cost vs benefit tradeoffs

**Key Insights for Portfolio Managers:**
- Protection is cheapest when VIX is low (contrary to intuition)
- VRP (volatility risk premium) is a key timing signal
- Dynamic hedging reduces drag in normal markets
- Tail risk parity naturally overweights safer assets
- Term structure signals can improve timing